In [ ]:
import sysfrom pathlib import Path# Handle multiple working directory scenarios# Case 1: Running from project root# Case 2: Running from notebooks/ directory# Case 3: Running as a Jupyter notebookcurrent_file = Path(__file__).resolve() if '__file__' in dir() else Path.cwd()if current_file.is_file():    # We're running a notebook file    notebook_dir = current_file.parent    project_root = notebook_dir.parentelse:    # We're in a directory    project_root = Path.cwd()    # If we're in notebooks/ subdirectory, go up one level    if project_root.name == 'notebooks':        project_root = project_root.parent# Add project root to path if not already thereif str(project_root) not in sys.path:    sys.path.insert(0, str(project_root))# Verify src module existssrc_path = project_root / 'src'if not src_path.exists():    raise RuntimeError(f"Could not find src/ directory. Project root: {project_root}")print(f"✓ Added {project_root} to sys.path")

# Training and Generating with TinyGPT

This notebook demonstrates autoregressive language modeling with GPT.

We'll:
1. Train GPT on a toy corpus
2. Observe how loss converges
3. Generate text with different temperatures
4. Compare greedy vs stochastic sampling

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from src.transformer.gpt import TinyGPT
from src.common.tokenizer import CharTokenizer

torch.manual_seed(42)
print("✓ Imports successful")

## Part 1: Setup and Data Preparation

In [ ]:
corpus = "the quick brown fox jumps over the lazy dog"
tokenizer = CharTokenizer(corpus)

print(f"Corpus: '{corpus}'")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Unique characters: {tokenizer.chars}")

token_ids = torch.tensor([tokenizer.encode(corpus)])
print(f"\nToken shape: {token_ids.shape}")
print(f"Tokens: {token_ids[0].tolist()}")

## Part 2: Create and Train GPT

In [ ]:
gpt = TinyGPT(
    vocab_size=tokenizer.vocab_size,
    d_model=64,
    n_layers=2,
    n_heads=4
)

print(f"Model created")
print(f"Total parameters: {sum(p.numel() for p in gpt.parameters()):,}")

optimizer = torch.optim.Adam(gpt.parameters(), lr=0.01)
num_epochs = 100
losses = []

print(f"\nTraining for {num_epochs} epochs...")
print(f"{'Epoch':<8} {'Loss':<12}")
print("-" * 20)

for epoch in range(num_epochs):
    optimizer.zero_grad()
    logits = gpt(token_ids)
    loss = F.cross_entropy(
        logits[:, :-1].reshape(-1, gpt.vocab_size),
        token_ids[:, 1:].reshape(-1)
    )
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    
    if epoch % 20 == 0:
        print(f"{epoch:<8} {loss.item():<12.4f}")

print(f"{num_epochs:<8} {losses[-1]:<12.4f}")
print(f"✓ Training complete")

## Part 3: Visualize Training Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(losses, linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss (Full)')
ax1.grid(True, alpha=0.3)

ax2.plot(losses[-20:], linewidth=2, color='orange')
ax2.set_xlabel('Epoch (last 20)')
ax2.set_ylabel('Loss')
ax2.set_title('Training Loss (Final Phase)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Starting loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Improvement: {(losses[0] - losses[-1]):.4f}")

## Part 4: Generate Text

In [ ]:
prompt = "the"
prompt_tokens = torch.tensor([tokenizer.encode(prompt)])

print(f"Prompt: '{prompt}'")
print("\n" + "="*60)
print("GENERATION WITH DIFFERENT TEMPERATURES")
print("="*60)

for temp in [0, 0.5, 1.0, 2.0]:
    with torch.no_grad():
        generated = gpt.generate(prompt_tokens, max_new_tokens=20, temperature=temp)
    
    text = tokenizer.decode(generated[0].tolist())
    print(f"\nTemperature {temp}: '{text}'")

## Part 5: Understanding Temperature

In [ ]:
print("Temperature Controls Sampling Randomness:")
print()
print("Formula: probs = softmax(logits / temperature)")
print()
print("T=0.0:   Greedy (always argmax) → deterministic, repetitive")
print("T=0.5:   Cold → favor likely tokens, less diversity")
print("T=1.0:   Normal → balanced diversity and coherence")
print("T=2.0:   Hot → uniform distribution, more diverse")
print()

logits = torch.tensor([[1.0, 2.0, 0.5]])
print("Example: logits = [1.0, 2.0, 0.5]")
for temp in [0.5, 1.0, 2.0]:
    probs = F.softmax(logits / temp, dim=-1)[0]
    print(f"T={temp}: [{probs[0]:.3f}, {probs[1]:.3f}, {probs[2]:.3f}]")

## Summary

**Autoregressive Generation with GPT:**

✓ Training: Next-token prediction from left context only  
✓ Causal Attention: Blocks future positions  
✓ Generation: Sequential left-to-right  
✓ Temperature: Controls diversity  

**Why This Works:**  
Training sees [t0, t1, ...] and predicts t_next.  
Inference does the same: given left context, sample next token.  
Training and inference are aligned — no distributional mismatch.